# Data Import and Feature Engineering

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
# user settings
INPUT_FILES = {
    "BTC": "data/raw/BTCUSDT_1h.csv",
    "ETH": "data/raw/ETHUSDT_1h.csv",
    "XRP": "data/raw/XRPUSDT_1h.csv",
    "SOL": "data/raw/SOLUSDT_1h.csv",
}

OUTPUT_DIR = "data/engineered_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPS = 1e-12

## Helper Indicator Functions

In [3]:
def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()

    rs = avg_gain / (avg_loss + EPS)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(close: pd.Series,
                 fast: int = 12,
                 slow: int = 26,
                 signal: int = 9):
    ema_fast = close.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_slow = close.ewm(span=slow, adjust=False, min_periods=slow).mean()
    macd_line = ema_fast - ema_slow
    macd_signal = macd_line.ewm(span=signal, adjust=False, min_periods=signal).mean()
    macd_hist = macd_line - macd_signal
    return macd_line, macd_signal, macd_hist


def compute_atr(high: pd.Series,
                low: pd.Series,
                close: pd.Series,
                window: int = 14) -> pd.Series:
    prev_close = close.shift(1)

    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()

    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = true_range.rolling(window=window, min_periods=window).mean()
    return atr


def rolling_zscore(series: pd.Series, window: int) -> pd.Series:
    mean = series.rolling(window, min_periods=window).mean()
    std = series.rolling(window, min_periods=window).std()
    return (series - mean) / (std + EPS)

## Feature Engineering

### Load and Standardise

In [4]:
def load_and_standardize(file_path: str, asset_name: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    keep_cols = [
        "open_time", "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"
    ]
    df = df[keep_cols].copy()

    # fix timestamp properly (NO WARNING VERSION)
    df["open_time"] = pd.to_numeric(df["open_time"], errors="coerce")

    # try both conversions
    dt_ms = pd.to_datetime(df["open_time"], unit="ms", utc=True, errors="coerce")
    dt_us = pd.to_datetime(df["open_time"], unit="us", utc=True, errors="coerce")

    # use whichever is valid (ms first, then fallback to us)
    df["datetime"] = dt_ms.fillna(dt_us)

    # clean datetime
    df = df.dropna(subset=["datetime"]).copy()
    df = df.sort_values("datetime").reset_index(drop=True)

    # convert numeric columns
    num_cols = [
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"
    ]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # basic cleaning
    df = df.drop_duplicates(subset=["datetime"]).reset_index(drop=True)

    df = df.dropna(subset=["open", "high", "low", "close"])

    df = df[
        (df["open"] > 0) &
        (df["high"] > 0) &
        (df["low"] > 0) &
        (df["close"] > 0)
    ]

    df["asset"] = asset_name

    return df

### Create Features

In [5]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # 1. basic price transforms
    # --------------------------------------------------
    out["log_close"] = np.log(out["close"])
    out["log_open"] = np.log(out["open"])

    # realised bar return (known at end of bar t)
    out["ret_1h_close"] = out["close"].pct_change(1)
    out["log_ret_1h_close"] = np.log(out["close"] / out["close"].shift(1))

    # multi-horizon momentum
    for h in [3, 6, 12, 24, 48, 72]:
        out[f"ret_{h}h"] = out["close"].pct_change(h)
        out[f"log_ret_{h}h"] = np.log(out["close"] / out["close"].shift(h))

    # 2. candlestick features (scale-free)
    # --------------------------------------------------
    max_oc = np.maximum(out["open"], out["close"])
    min_oc = np.minimum(out["open"], out["close"])

    out["range_pct"] = (out["high"] - out["low"]) / (out["open"] + EPS)
    out["body_pct"] = (out["close"] - out["open"]) / (out["open"] + EPS)
    out["body_abs_pct"] = out["body_pct"].abs()
    out["upper_shadow_pct"] = (out["high"] - max_oc) / (out["open"] + EPS)
    out["lower_shadow_pct"] = (min_oc - out["low"]) / (out["open"] + EPS)
    out["body_to_range"] = (out["close"] - out["open"]) / ((out["high"] - out["low"]) + EPS)

    # close location within the bar: near high or near low
    out["close_loc"] = (out["close"] - out["low"]) / ((out["high"] - out["low"]) + EPS)

    # directional sign
    out["bar_up"] = (out["close"] > out["open"]).astype(int)
    out["bar_down"] = (out["close"] < out["open"]).astype(int)

    # gap from previous close to current open
    out["gap_open_prev_close"] = out["open"] / out["close"].shift(1) - 1

    # 3. trend / moving average features
    # --------------------------------------------------
    for w in [6, 12, 24, 48, 72]:
        out[f"sma_{w}"] = out["close"].rolling(w, min_periods=w).mean()
        out[f"ema_{w}"] = out["close"].ewm(span=w, adjust=False, min_periods=w).mean()

        out[f"close_to_sma_{w}"] = out["close"] / (out[f"sma_{w}"] + EPS) - 1
        out[f"close_to_ema_{w}"] = out["close"] / (out[f"ema_{w}"] + EPS) - 1

    # ma spread features
    out["sma_cross_6_24"] = out["sma_6"] / (out["sma_24"] + EPS) - 1
    out["sma_cross_12_48"] = out["sma_12"] / (out["sma_48"] + EPS) - 1
    out["ema_cross_12_24"] = out["ema_12"] / (out["ema_24"] + EPS) - 1

    # 4. volatility / risk features
    # --------------------------------------------------
    for w in [6, 24, 72]:
        out[f"vol_{w}h"] = out["log_ret_1h_close"].rolling(w, min_periods=w).std()

    # downside volatility
    neg_ret = out["log_ret_1h_close"].where(out["log_ret_1h_close"] < 0, 0)
    for w in [24, 72]:
        out[f"downside_vol_{w}h"] = neg_ret.rolling(w, min_periods=w).std()

    # ATR scaled by price
    out["atr_14"] = compute_atr(out["high"], out["low"], out["close"], window=14)
    out["atr_14_pct"] = out["atr_14"] / (out["close"] + EPS)

    # 5. momentum oscillators
    # --------------------------------------------------
    out["rsi_14"] = compute_rsi(out["close"], window=14)

    macd_line, macd_signal, macd_hist = compute_macd(out["close"])
    out["macd_line"] = macd_line
    out["macd_signal"] = macd_signal
    out["macd_hist"] = macd_hist

    # scale MACD by price to improve comparability
    out["macd_line_pct"] = out["macd_line"] / (out["close"] + EPS)
    out["macd_signal_pct"] = out["macd_signal"] / (out["close"] + EPS)
    out["macd_hist_pct"] = out["macd_hist"] / (out["close"] + EPS)

    # 6. volume / market activity features
    # --------------------------------------------------
    out["log_volume"] = np.log1p(out["volume"])
    out["log_quote_volume"] = np.log1p(out["quote_asset_volume"])
    out["log_trades"] = np.log1p(out["number_of_trades"])

    # relative volume / abnormal activity
    out["vol_z_24"] = rolling_zscore(out["log_volume"], 24)
    out["quote_vol_z_24"] = rolling_zscore(out["log_quote_volume"], 24)
    out["trades_z_24"] = rolling_zscore(out["log_trades"], 24)

    out["vol_ratio_24"] = out["volume"] / (out["volume"].rolling(24, min_periods=24).mean() + EPS)
    out["quote_vol_ratio_24"] = out["quote_asset_volume"] / (
        out["quote_asset_volume"].rolling(24, min_periods=24).mean() + EPS
    )

    # taker-buy imbalance proxy
    out["taker_buy_base_share"] = out["taker_buy_base_asset_volume"] / (out["volume"] + EPS)
    out["taker_buy_quote_share"] = out["taker_buy_quote_asset_volume"] / (out["quote_asset_volume"] + EPS)
    out["order_flow_imbalance"] = 2 * out["taker_buy_base_share"] - 1

    # 7. time-of-day / day-of-week seasonality
    # --------------------------------------------------
    out["hour"] = out["datetime"].dt.hour
    out["dayofweek"] = out["datetime"].dt.dayofweek

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["dayofweek"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dayofweek"] / 7)

    # 8. forward returns aligned with execution rule
    # --------------------------------------------------
    # signal made at end of t
    # enter at open of t+1
    # hold until open of t+2
    out["fwd_ret_open_1h"] = out["open"].shift(-2) / out["open"].shift(-1) - 1
    out["fwd_log_ret_open_1h"] = np.log(out["open"].shift(-2) / out["open"].shift(-1))

    # binary labels for classification
    out["label_up"] = (out["fwd_ret_open_1h"] > 0).astype(int)
    out["label_down"] = (out["fwd_ret_open_1h"] < 0).astype(int)

    # optional three-way raw direction label
    out["direction_3class"] = np.select(
        [
            out["fwd_ret_open_1h"] > 0,
            out["fwd_ret_open_1h"] < 0
        ],
        [1, -1],
        default=0
    )

    # 9. keep raw execution prices needed later for backtest
    # --------------------------------------------------
    out["entry_open_t1"] = out["open"].shift(-1)
    out["exit_open_t2"] = out["open"].shift(-2)

    # 10. final cleaning
    # --------------------------------------------------
    # Replace inf values from logs/ratios if any
    out = out.replace([np.inf, -np.inf], np.nan)

    return out

In [6]:
def build_feature_dataset(file_path: str, asset_name: str) -> pd.DataFrame:
    raw = load_and_standardize(file_path, asset_name)
    feat = create_features(raw)

    # columns to keep for modeling and later backtest
    keep_cols = [
        "datetime", "asset",
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",

        # backtest alignment columns
        "entry_open_t1", "exit_open_t2",
        "fwd_ret_open_1h", "fwd_log_ret_open_1h",
        "label_up", "label_down", "direction_3class",

        # price/return features
        "ret_1h_close", "log_ret_1h_close",
        "ret_3h", "log_ret_3h",
        "ret_6h", "log_ret_6h",
        "ret_12h", "log_ret_12h",
        "ret_24h", "log_ret_24h",
        "ret_48h", "log_ret_48h",
        "ret_72h", "log_ret_72h",

        # candlestick features
        "range_pct", "body_pct", "body_abs_pct",
        "upper_shadow_pct", "lower_shadow_pct",
        "body_to_range", "close_loc",
        "bar_up", "bar_down",
        "gap_open_prev_close",

        # trend features
        "close_to_sma_6", "close_to_sma_12", "close_to_sma_24",
        "close_to_sma_48", "close_to_sma_72",
        "close_to_ema_6", "close_to_ema_12", "close_to_ema_24",
        "close_to_ema_48", "close_to_ema_72",
        "sma_cross_6_24", "sma_cross_12_48", "ema_cross_12_24",

        # volatility features
        "vol_6h", "vol_24h", "vol_72h",
        "downside_vol_24h", "downside_vol_72h",
        "atr_14_pct",

        # oscillators
        "rsi_14",
        "macd_line_pct", "macd_signal_pct", "macd_hist_pct",

        # volume / activity
        "log_volume", "log_quote_volume", "log_trades",
        "vol_z_24", "quote_vol_z_24", "trades_z_24",
        "vol_ratio_24", "quote_vol_ratio_24",
        "taker_buy_base_share", "taker_buy_quote_share",
        "order_flow_imbalance",

        # time features
        "hour_sin", "hour_cos", "dow_sin", "dow_cos"
    ]

    feat = feat[keep_cols].copy()

    # drop rows with missing engineered features / targets
    feat = feat.dropna().reset_index(drop=True)

    return feat

### Run Script

In [7]:
all_dfs = []

for asset, file_path in INPUT_FILES.items():
    df_feat = build_feature_dataset(file_path, asset)
    all_dfs.append(df_feat)

    out_path = os.path.join(OUTPUT_DIR, f"{asset}_features.csv")
    df_feat.to_csv(out_path, index=False)

    print(f"{asset}: saved {len(df_feat):,} rows to {out_path}")

BTC: saved 43,736 rows to data/engineered_features/BTC_features.csv
ETH: saved 43,736 rows to data/engineered_features/ETH_features.csv
XRP: saved 43,736 rows to data/engineered_features/XRP_features.csv
SOL: saved 43,736 rows to data/engineered_features/SOL_features.csv
